In [ ]:
from matplotlib import pyplot as plt
from matplotlib import colors as colors
import contextily as ctx

from extractosm.transit import (
    extract_all_transit_stops,
    extract_all_transit_routes,
    extract_transit_network,
)

from extractosm.pois import get_osm_features

In [ ]:
bbox = (6.5,46.5,6.8,46.6)
features = get_osm_features(bbox, {"amenity": True}, osm_pbf_path="data/lausanne-filtered.osm.pbf")

In [ ]:
all_stops = extract_all_transit_stops(
    osm_pbf_path="data/lausanne-filtered.osm.pbf",
    output_path="data/all_stops.geoparquet",
)

In [ ]:
all_stops

In [ ]:
ax = all_stops.to_crs("EPSG:3857").plot(
    column="stop_type", figsize=(10, 10), legend=True
)
ax.set_xlim(7.3e5, 7.45e5)
ax.set_ylim(5.86e6, 5.87e6)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.CH, zoom=14)
plt.title("Transit stops around Lausanne")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
# limit the extent to the area of interest
plt.show()

In [ ]:
routes = extract_all_transit_routes(
    osm_pbf_path="data/lausanne-filtered.osm.pbf",
    output_path="data/all_routes.geoparquet",
    route_types=["bus", "tram", "subway", "trolleybus", "light_rail"],
    group_by="route_master",
    include_way_ids=True
)

In [ ]:
# Explode the data frame based on the way IDs and get the list of all route_master_names for each way ID
routes.explode("way_ids").reset_index(drop=True).groupby("way_ids").agg({"route_master_name": list})

In [ ]:
routes[routes["network"] == "Mobilis"].head()

In [ ]:
ax = (
    routes[routes["network"] == "Mobilis"]
    .to_crs("EPSG:3857")
    .plot(column="route_master_name", figsize=(10, 10), legend=True,
    legend_kwds={
        "loc": "upper left",
        "bbox_to_anchor": (1.02, 1),
        "title": "Route master name",
        "frameon": True,
    },)
)
ax.set_xlim(7.3e5, 7.45e5)
ax.set_ylim(5.86e6, 5.87e6)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.CH, zoom=14)
plt.title("Mobilis transit routes around Lausanne")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
# limit the extent to the area of interest
plt.show()

In [ ]:
routes[routes["route_master_ref"] == "m2"]

In [ ]:
routes[routes["network"] == "Mobilis"].head()

In [ ]:
network = extract_transit_network(
    osm_pbf_path="data/lausanne-filtered.osm.pbf",
    exclude_networks=["Flixbus"],
    route_types=["bus", "tram", "subway", "trolleybus", "train", "light_transit"],
    include_route_ids=True,
    include_stop_ids=True,
)